# Notebook 2: Text Preprocessing & TF-IDF

This notebook answers: **how do we turn raw article text into numbers a model can learn from?**

Covers ADRs 004–007: stopword choice, TF-IDF design, bigrams, title+content concatenation, and artifact persistence.

---

## The Core Problem

Machine learning models only understand numbers. An article like:

> *"The Federal Reserve raised interest rates by 0.25 percent"*

...means nothing to a model until it's a vector of numbers. This notebook shows you exactly how that conversion happens.

```
Raw article text
      │
      ▼
 Preprocessing      ← lowercase, remove punctuation, drop stopwords
      │
      ▼
 TF-IDF Vectorizer  ← convert words to importance scores
      │
      ▼
 Numeric vector     ← [0.0, 0.42, 0.0, 0.18, 0.0, 0.61, ...] (50,000 values)
```

In [ ]:
import re
import pickle
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords

sns.set_theme(style='whitegrid', palette='muted')

# Load the cleaned articles produced by transform.py
df = pd.read_parquet('../data/processed/articles_clean.parquet')
print(f'Loaded {len(df):,} cleaned articles')
print(f'Columns: {df.columns.tolist()}')

---

## ADR-004: Text Preprocessing — Why and What

### Why preprocess at all?

Raw text has a lot of noise that doesn't help distinguish topics:
- Punctuation: `.`, `,`, `!` carry no topic signal
- Case: "Federal" and "federal" should be the same word
- Stopwords: "the", "and", "is" appear in every article equally — they tell us nothing about topic

In [ ]:
# Show a real before/after example from our dataset
sample_raw = df['content'].iloc[0][:500]
sample_clean = df['text_clean'].iloc[0][:500]

print('=== BEFORE preprocessing ===')
print(sample_raw)
print()
print('=== AFTER preprocessing ===')
print(sample_clean)

In [ ]:
# Show NLTK's stopword list — this is why we chose NLTK over a hand-rolled list
STOPWORDS = set(stopwords.words('english'))
print(f'NLTK stopwords: {len(STOPWORDS)} words')
print()
print(sorted(STOPWORDS))

In [ ]:
# Demonstrate the preprocessing steps one at a time on a single sentence
sentence = "The Federal Reserve's interest-rate decision will affect millions of Americans!"

step1 = sentence.lower()
step2 = re.sub(r'http\S+|www\S+', ' ', step1)
step3 = re.sub(r'[^a-z\s]', ' ', step2)
step4 = re.sub(r'\s+', ' ', step3).strip()
tokens = [w for w in step4.split() if w not in STOPWORDS and len(w) > 1]
step5 = ' '.join(tokens)

print(f'Original:              "{sentence}"')
print(f'Step 1 - Lowercase:    "{step1}"')
print(f'Step 2 - Remove URLs:  "{step2}"')
print(f'Step 3 - Alpha only:   "{step3.strip()}"')
print(f'Step 4 - Whitespace:   "{step4}"')
print(f'Step 5 - Stopwords:    "{step5}"')
print(f'\nRemoved stopwords: {[w for w in step4.split() if w in STOPWORDS]}')

> **Interview Talking Point (ADR-004):**  
> *"I used NLTK's 179-word stopword list rather than writing my own. It covers inflected forms like 'ourselves' and contractions like 'wouldn't' that a hand-rolled list would miss. The preprocessing pipeline lowercases, strips non-alphabetic characters, and removes stopwords — reducing vocabulary noise before vectorization."*

---

## ADR-006: Why Concatenate Title + Content?

Before we vectorize, we combine the title and article body into one string. Here's why:

In [ ]:
# Titles are dense with topic signal — they're human-written summaries
print('Example titles from our dataset:')
for title in df['title'].sample(8, random_state=42).values:
    print(f'  • {title}')

In [ ]:
# The title's keywords now appear at least twice in the combined text:
# once from the title, possibly again in the body
# This gives them a higher TF score — which is exactly what we want

sample = df.iloc[0]
title_words = set(sample['title'].lower().split())
content_words = sample['content'].lower().split()

# How often do title words appear in the content?
overlap = [(w, content_words.count(w)) for w in title_words if w not in STOPWORDS and len(w) > 2]
overlap = sorted(overlap, key=lambda x: x[1], reverse=True)[:8]

print(f'Title: "{sample["title"]}"')
print(f'\nTitle keywords and how often they appear in the content body:')
for word, count in overlap:
    print(f'  "{word}": {count} times in content  →  combined text count: {count + 1}')

> **Interview Talking Point (ADR-006):**  
> *"I prepend the article title to the content before vectorizing. Titles are written by editors to summarize the topic — they're dense with signal. Concatenating them gives title keywords a slightly higher term frequency in the TF-IDF vector, which biases the model toward the most topic-relevant words."*

---

## ADR-005: TF-IDF — The Core Vectorization Step

### What is TF-IDF?

TF-IDF stands for **Term Frequency × Inverse Document Frequency**. It scores how *important* a word is to a specific document relative to the entire corpus.

**Intuition:** A word is important if it appears often in *this* article but rarely across *all* articles.

- "federal reserve" appears a lot in an economics article, rarely elsewhere → **high score**
- "said" appears in every article equally → **near zero score**

In [ ]:
# ── Build intuition with a tiny toy example first ──────────────────────────

toy_corpus = [
    "the federal reserve raised interest rates",        # economics article
    "interest rates affect mortgage and housing market", # housing article
    "the president signed the new tax bill into law",   # politics article
    "congress passed the tax reform legislation today", # politics article
]

toy_vec = TfidfVectorizer()
toy_matrix = toy_vec.fit_transform(toy_corpus)
toy_df = pd.DataFrame(
    toy_matrix.toarray().round(2),
    columns=toy_vec.get_feature_names_out(),
    index=['econ article', 'housing article', 'politics 1', 'politics 2']
)

print('TF-IDF matrix — each row is an article, each column is a word:')
print('(Only showing words that score > 0 somewhere)')
toy_df

In [ ]:
# Notice: 
# - "the" scores 0 or near 0 everywhere (appears in too many docs — high IDF penalty)
# - "federal" and "reserve" only appear in doc 0, so they score HIGH there
# - "tax" appears in docs 2 and 3, scoring high in both but not others

print('Key observations:')
print(f'  "federal" in econ article:    {toy_df.loc["econ article", "federal"]:.2f}  ← unique to this doc → high score')
print(f'  "the"  in econ article:       {toy_df.loc["econ article", "the"]:.2f}  ← appears everywhere → low score')
print(f'  "interest" in econ:           {toy_df.loc["econ article", "interest"]:.2f}  ← appears in 2 docs → medium score')
print(f'  "interest" in housing:        {toy_df.loc["housing article", "interest"]:.2f}  ← same word, same score')

### The Math (kept simple)

```
TF(word, doc)  = count of word in this doc / total words in this doc

IDF(word)      = log( total docs / docs containing word )
                 ↑ rare word → big number  |  common word → small number

TF-IDF(word, doc) = TF × IDF
```

We use `sublinear_tf=True`, which replaces TF with `log(1 + TF)`. This dampens the effect of a word that appears 50 times vs 5 times — both are "frequent", the raw count difference is less meaningful.

In [ ]:
# Demonstrate sublinear_tf — why log damping matters
raw_counts = [1, 5, 10, 50, 100]
print('Word count  | Raw TF (linear) | log(1 + count) (sublinear)')
print('-' * 55)
for c in raw_counts:
    print(f'{c:>10}  | {c/100:>15.2f}  | {np.log1p(c):>26.3f}')

print('\nWith sublinear_tf, a word appearing 100x is only ~2x more important')
print('than one appearing 5x, rather than 20x more important.')

In [ ]:
# ── Now look at our REAL TF-IDF matrix ───────────────────────────────────────
vectorizer = pickle.load(open('../data/processed/tfidf_vectorizer.pkl', 'rb'))
tfidf_matrix = sp.load_npz('../data/processed/tfidf_matrix.npz')

print(f'Matrix shape: {tfidf_matrix.shape[0]:,} articles × {tfidf_matrix.shape[1]:,} features')
print(f'Sparsity:     {100*(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0]*tfidf_matrix.shape[1])):.1f}% zeros')
print(f'Non-zero values stored: {tfidf_matrix.nnz:,}  (everything else is implicitly 0)')

In [ ]:
# Look at the top TF-IDF words for a single article
article_idx = 5  # pick any article index

feature_names = vectorizer.get_feature_names_out()
article_vector = tfidf_matrix[article_idx].toarray().flatten()

# Get the top 15 scoring terms
top_indices = article_vector.argsort()[::-1][:15]
top_terms = [(feature_names[i], round(article_vector[i], 4)) for i in top_indices]

print(f'Article title: "{df["title"].iloc[article_idx]}"')
print(f'Publication:   {df["publication"].iloc[article_idx]}')
print(f'\nTop 15 TF-IDF terms (most important words for this article):')
for term, score in top_terms:
    bar = '█' * int(score * 80)
    print(f'  {term:<30} {score:.4f}  {bar}')

### Why bigrams? (`ngram_range=(1, 2)`)

In [ ]:
# Bigrams capture two-word phrases that mean something specific together
# that the individual words don't capture alone

bigram_examples = [
    ('white', 'house', 'white house → the presidency'),
    ('interest', 'rate', 'interest rate → economics/Fed'),
    ('climate', 'change', 'climate change → environment'),
    ('supreme', 'court', 'supreme court → judiciary'),
    ('north', 'korea', 'north korea → foreign policy'),
]

print('Bigrams capture topic-specific phrases that unigrams miss:')
print()
for w1, w2, meaning in bigram_examples:
    print(f'  "{w1}" alone: generic')
    print(f'  "{w2}" alone: generic')
    print(f'  "{w1} {w2}": specific → {meaning}')
    print()

# Check if our vectorizer captured these
vocab = vectorizer.vocabulary_
print('Bigrams found in our vocabulary:')
for w1, w2, _ in bigram_examples:
    phrase = f'{w1} {w2}'
    found = phrase in vocab
    print(f'  "{phrase}": {"✓ in vocabulary" if found else "✗ not found (may have been filtered)"}')

In [ ]:
# Show the vocabulary composition — how many unigrams vs bigrams?
vocab_terms = list(vectorizer.vocabulary_.keys())
unigrams = [t for t in vocab_terms if ' ' not in t]
bigrams  = [t for t in vocab_terms if ' ' in t]

print(f'Total vocabulary: {len(vocab_terms):,} terms')
print(f'  Unigrams (single words): {len(unigrams):,}  ({100*len(unigrams)/len(vocab_terms):.0f}%)')
print(f'  Bigrams  (two-word):     {len(bigrams):,}  ({100*len(bigrams)/len(vocab_terms):.0f}%)')
print(f'\nSample bigrams in vocabulary:')
import random
random.seed(42)
print(' | '.join(random.sample(bigrams, 20)))

> **Interview Talking Point (ADR-005):**  
> *"I used TF-IDF with sublinear term frequency scaling and bigrams. TF-IDF rewards words that are distinctive to a specific article, penalizing words common across all articles. Sublinear scaling prevents a word appearing 100 times from dominating a word appearing 10 times. Bigrams like 'interest rate' and 'white house' capture topic-specific phrases that single words miss."*

---

## ADR-007: Why Save Artifacts to Disk?

We save three files after transform.py runs. Here's why each one matters:

In [ ]:
from pathlib import Path

artifacts = [
    '../data/processed/articles_clean.parquet',
    '../data/processed/tfidf_matrix.npz',
    '../data/processed/tfidf_vectorizer.pkl',
]

print('Saved artifacts:')
for path in artifacts:
    size_mb = Path(path).stat().st_size / 1e6
    print(f'  {Path(path).name:<35} {size_mb:>7.1f} MB')

In [ ]:
# The most critical artifact is the VECTORIZER — here's why
# At inference time (predicting topic for a NEW article), you must transform
# it with the SAME vocabulary and IDF weights as the training data.
# If you refit on new data, word scores change and the model's predictions break.

new_article = "The Federal Reserve announced a new interest rate policy today."

# Correct: use the saved vectorizer from training
new_vector = vectorizer.transform([new_article])
print(f'New article transformed correctly:')
print(f'  Input: "{new_article}"')
print(f'  Output: sparse vector with {new_vector.nnz} non-zero values out of {new_vector.shape[1]:,} features')

print()
print('Top terms for this new article:')
feature_names = vectorizer.get_feature_names_out()
vec_arr = new_vector.toarray().flatten()
top_idx = vec_arr.argsort()[::-1][:8]
for i in top_idx:
    if vec_arr[i] > 0:
        print(f'  {feature_names[i]:<25} {vec_arr[i]:.4f}')

> **Interview Talking Point (ADR-007):**  
> *"I save the fitted TF-IDF vectorizer to disk alongside the matrix. This is critical for production correctness — at inference time, a new article must be transformed using the exact same vocabulary and IDF weights as training data. If you refit the vectorizer, word scores shift and predictions break. Saving it ensures transform consistency across train and serve."*

---

## Summary

```
Raw text  →  [lowercase + remove punct + drop stopwords]  →  cleaned text
                                                                   │
                               [TF-IDF: sublinear, bigrams, 50k]  │
                                                                   ▼
                                             141k × 50k sparse matrix
                                             (99.4% zeros — efficient)
```

| Parameter | Value | Why |
|-----------|-------|-----|
| `sublinear_tf` | True | Dampen effect of high-frequency words |
| `ngram_range` | (1,2) | Capture two-word phrases |
| `max_features` | 50,000 | Limit memory and training time |
| `min_df` | 5 | Drop rare words (likely typos) |
| `max_df` | 0.95 | Drop near-universal words |